# Atelier Scikit-learn — Prédiction de l'état des capteurs IoT

Objectif : construire un modèle de Machine Learning capable de prédire automatiquement l'**état** d'un capteur (OK, ALERTE, ERREUR) à partir de ses mesures (température, humidité, pression, consommation).

Workflow suivi : Dataset → Chargement → Exploration → Nettoyage → X / y → Train / Test → Prétraitement → Modèle → fit() → predict() → Évaluation → Sauvegarde → Chargement → Réutilisation.

## Partie 0 — Mise en place de l'environnement

### 4) Import des librairies

In [19]:
import matplotlib
print(matplotlib.__version__)
import matplotlib.pyplot as plt

3.11.1


In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (7, 5)


### 5) Import du dataset dans `df`

In [21]:
df = pd.read_csv("../data/mesures_capteurs.csv")
df.head()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


### 6) Exploration du dataframe

In [22]:
print("Dimensions :", df.shape)
df.info()

Dimensions : (605, 9)
<class 'pandas.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     605 non-null    str    
 1   date_heure    605 non-null    str    
 2   id_capteur    605 non-null    str    
 3   batiment      605 non-null    str    
 4   temperature   599 non-null    float64
 5   humidite      600 non-null    float64
 6   pression      600 non-null    float64
 7   consommation  600 non-null    float64
 8   etat          601 non-null    str    
dtypes: float64(4), str(5)
memory usage: 42.7 KB


In [23]:
df.describe(include="all")

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
count,605,605,605,605,599.000000,600.00000,600.000000,600.000000,601
unique,600,600,12,4,NaN,NaN,NaN,NaN,3
top,M0026,2026-01-06 01:00:00,C002,B001,NaN,NaN,NaN,NaN,OK
freq,2,2,52,153,NaN,NaN,NaN,NaN,567
mean,NaN,NaN,NaN,NaN,24.878314,64.92620,1012.221900,208.675417,NaN
std,NaN,NaN,NaN,NaN,4.059576,10.76905,10.599042,72.243567,NaN
min,NaN,NaN,NaN,NaN,-18.500000,28.52000,850.000000,18.120000,NaN
25%,NaN,NaN,NaN,NaN,22.570000,58.17250,1006.790000,160.177500,NaN
50%,NaN,NaN,NaN,NaN,24.860000,65.37500,1012.855000,206.150000,NaN
75%,NaN,NaN,NaN,NaN,27.275000,71.61500,1017.827500,254.127500,NaN


In [24]:
df.isna().sum()

id_mesure       0
date_heure      0
id_capteur      0
batiment        0
temperature     6
humidite        5
pression        5
consommation    5
etat            4
dtype: int64

In [25]:
df["etat"].value_counts()

etat
OK        567
ALERTE     29
ERREUR      5
Name: count, dtype: int64

**Constat :** le dataset contient 605 lignes. La cible `etat` est fortement **déséquilibrée** : très majoritairement `OK`, avec beaucoup moins de cas `ALERTE` et très peu de cas `ERREUR`. On note aussi des valeurs manquantes sur plusieurs colonnes, dont la cible elle-même — à traiter avant la modélisation.

## Partie 1 — Gestion des doublons

### 1) Vérifier l'existence de doublons

In [26]:
nb_doublons = df.duplicated().sum()
print(f"Nombre de doublons : {nb_doublons}")
df[df.duplicated()]

Nombre de doublons : 5


,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
183,M0599,2026-01-29 22:00:00,C011,B004,22.02,68.09,1005.90,227.81,OK
231,M0026,2026-01-06 01:00:00,C002,B001,22.87,77.99,1010.15,213.19,OK
355,M0147,2026-01-11 02:00:00,C003,B001,26.14,84.97,1003.59,142.31,OK
538,M0456,2026-01-23 23:00:00,C012,B004,20.68,72.69,1022.77,309.01,OK
539,M0302,2026-01-17 13:00:00,C002,B001,22.05,58.26,1007.30,140.42,OK


### 2) Supprimer les doublons et vérifier

In [27]:
df = df.drop_duplicates().reset_index(drop=True)
print("Nombre de doublons après suppression :", df.duplicated().sum())
print("Nouvelles dimensions :", df.shape)

Nombre de doublons après suppression : 0
Nouvelles dimensions : (600, 9)


## Partie 2 — Sélection de y (cible) et X (caractéristiques)

On retire d'abord les lignes où la cible `etat` est manquante : on ne peut pas entraîner ni évaluer un modèle supervisé sans étiquette connue.

In [28]:
df = df.dropna(subset=["etat"]).reset_index(drop=True)
print("Dimensions après suppression des etats manquants :", df.shape)

Dimensions après suppression des etats manquants : (596, 9)


### 1) Définition de X et y

In [29]:
features = ["temperature", "humidite", "pression", "consommation"]
X = df[features]
y = df["etat"]

### 2) Cinq premières lignes de X et de y

In [30]:
X.head()

,temperature,humidite,pression,consommation
0,25.46,58.06,1008.95,287.28
1,24.00,79.73,993.39,116.20
2,25.82,54.47,1010.32,288.50
3,28.23,69.39,1019.62,136.65
4,20.58,53.80,1016.58,182.62


In [31]:
y.head()

0    OK
1    OK
2    OK
3    OK
4    OK
Name: etat, dtype: str

### 3) Type du problème de Machine Learning

C'est un problème d'**apprentissage supervisé de classification multi-classes** (3 classes : `OK`, `ALERTE`, `ERREUR`), puisque la cible `etat` est une variable catégorielle discrète et non une valeur numérique continue.

## Partie 3 — Découpage Train/Test

In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train :", X_train.shape, " X_test :", X_test.shape)
print("\nRépartition des classes (train) :")
print(y_train.value_counts(normalize=True).round(3))
print("\nRépartition des classes (test) :")
print(y_test.value_counts(normalize=True).round(3))

X_train : (476, 4)  X_test : (120, 4)

Répartition des classes (train) :
etat
OK        0.943
ALERTE    0.048
ERREUR    0.008
Name: proportion, dtype: float64

Répartition des classes (test) :
etat
OK        0.942
ALERTE    0.050
ERREUR    0.008
Name: proportion, dtype: float64


- `test_size=0.2` → 20 % des données réservées au test.
- `random_state=42` → garantit la **reproductibilité** du découpage.
- `stratify=y` → conserve les **mêmes proportions de classes** dans train et test que dans les données d'origine (important ici car `etat` est très déséquilibré).

## Partie 4 — Gestion des valeurs manquantes

### 1) Vérifier l'existence de valeurs manquantes

In [33]:
print("Valeurs manquantes X_train :")
print(X_train.isna().sum())
print("\nValeurs manquantes X_test :")
print(X_test.isna().sum())

Valeurs manquantes X_train :
temperature     5
humidite        4
pression        5
consommation    3
dtype: int64

Valeurs manquantes X_test :
temperature     1
humidite        1
pression        0
consommation    2
dtype: int64


### 2) Sélection de SimpleImputer avec la stratégie médiane

In [34]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

### 3) Pourquoi la médiane ?

La **médiane** est robuste aux valeurs extrêmes (outliers), contrairement à la moyenne qui peut être fortement tirée par des valeurs aberrantes — or on a observé dans l'atelier précédent des températures extrêmes (ex. -18,5 °C ou 58,7 °C). La médiane donne donc une estimation plus fiable de la 'valeur typique' pour remplacer les données manquantes.

### 4) Calcul des paramètres (médianes) sur X_train

In [35]:
imputer.fit(X_train)
pd.Series(imputer.statistics_, index=features)

temperature       24.90
humidite          65.38
pression        1012.30
consommation     206.59
dtype: float64

### 5) Transformation de X_train et X_test

In [36]:
X_train_imputed = pd.DataFrame(imputer.transform(X_train), columns=features, index=X_train.index)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=features, index=X_test.index)

print("Valeurs manquantes restantes (train) :", X_train_imputed.isna().sum().sum())
print("Valeurs manquantes restantes (test) :", X_test_imputed.isna().sum().sum())

Valeurs manquantes restantes (train) : 0
Valeurs manquantes restantes (test) : 0


**Important :** l'imputeur est calibré (`fit`) uniquement sur `X_train`, puis appliqué (`transform`) à la fois sur `X_train` et `X_test`, afin d'éviter toute **fuite de données (data leakage)** depuis l'ensemble de test.

## Partie 5 — Mise à l'échelle

### 1) Sélection de StandardScaler

In [37]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

### 2) Pourquoi standardiser ?

Le modèle **KNN** (utilisé en Partie 6) est basé sur des **distances** entre points. Si les variables n'ont pas la même échelle (ex. pression ~1000 hPa vs humidité ~0-100 %), les variables aux valeurs les plus grandes domineraient artificiellement le calcul de distance. La standardisation (moyenne 0, écart-type 1) met toutes les variables sur un pied d'égalité.

### 3) Calcul des paramètres (moyennes, écarts-types) sur X_train_imputed

In [38]:
scaler.fit(X_train_imputed)
pd.DataFrame({"moyenne": scaler.mean_, "ecart_type": scaler.scale_}, index=features)

,moyenne,ecart_type
temperature,24.963739,4.193040
humidite,64.850441,10.162792
pression,1012.076218,10.994184
consommation,210.153950,73.420923


### 4) Transformation de X_train_imputed et X_test_imputed

In [39]:
X_train_scaled = pd.DataFrame(scaler.transform(X_train_imputed), columns=features, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imputed), columns=features, index=X_test.index)

X_train_scaled.describe().round(2)

,temperature,humidite,pression,consommation
count,476.00,476.00,476.00,476.00
mean,-0.00,0.00,-0.00,0.00
std,1.00,1.00,1.00,1.00
min,-10.37,-2.86,-14.74,-2.62
25%,-0.55,-0.66,-0.49,-0.68
50%,-0.02,0.05,0.02,-0.05
75%,0.56,0.66,0.52,0.61
max,8.05,2.78,2.40,9.06


## Partie 6 — Entraînement et prédiction d'un modèle

### 1) et 2) Sélection et entraînement du modèle KNN (k=5)

In [40]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[object](3,)","['ALERTE','ERREUR','OK']"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


### 3) Prédiction sur l'ensemble de test

In [41]:
y_pred = model.predict(X_test_scaled)
y_pred[:10]

array(['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK'],
      dtype=object)

### 4) Affichage de quelques prédictions

In [42]:
resultats = pd.DataFrame({
    "y_test (reel)": y_test.values[:10],
    "y_pred (predit)": y_pred[:10]
})
resultats

,y_test (reel),y_pred (predit)
0,OK,OK
1,OK,OK
2,OK,OK
3,OK,OK
4,ALERTE,OK
5,OK,OK
6,OK,OK
7,OK,OK
8,OK,OK
9,OK,OK


### 5) Comparaison prédictions vs vraies valeurs

In [43]:
comparaison = pd.DataFrame({"reel": y_test.values, "predit": y_pred})
comparaison["correct"] = comparaison["reel"] == comparaison["predit"]
print(comparaison["correct"].value_counts())
comparaison.head(15)

correct
True     115
False      5
Name: count, dtype: int64


,reel,predit,correct
0,OK,OK,True
1,OK,OK,True
2,OK,OK,True
3,OK,OK,True
4,ALERTE,OK,False
5,OK,OK,True
6,OK,OK,True
7,OK,OK,True
8,OK,OK,True
9,OK,OK,True
